# Config

In [11]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [12]:
import pandas as pd
import os
from preprocess import clean_text
from translate import translator, gen_text_for_embedding
import time


# 1) Preprocesamiento de los datos


In [13]:
# 1) Cargar datos
path = "/tmp/data"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Preprocesar los datos

#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)

# 2) Traducción del texto

## Translator

In [14]:
df = df.iloc[:10]  # Subset para pruebas rápidas

In [15]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
df[list(cols.values())] = df[list(cols.keys())].applymap(trans.detect_and_translate)
end = time.time()
#Reseteo del contador de textos procesados
trans.reset_count()

#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Usando dispositivo: cuda
Procesados 0 textos.
Tiempo total de traducción: 14.56 segundos


In [16]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
#   para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
df = gen_text_for_embedding(df, cols)
#Guardado de resultados
#Voy a cambiar esto por csv
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)

savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Titulo_trad,Resumen_trad,keywords_trad,Facultad_del_Proyecto_trad,Depto_Persona_trad,text_for_embedding_translated
0,217.173.049-1.0,SI,,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,FACULTAD DE CIENCIAS SOCIALES,"DEPARTAMENTO DE ECONOMÍA, SIN INFORMACIÓN, ESC...",patterns of gender upbringing and socializatio...,general objectives: to describe the processes ...,,Faculty of Social Sciences,"department of economics, without information, ...",patterns of gender upbringing and socializatio...
1,218.201.002-1.0,SI,,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,FACULTAD DE ENFERMERÍA,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",cultural adaptation and validation of the life...,In order to evaluate the behaviors related to ...,"lifestyle, teens",Faculty of Nursing,"Department of Animal Science, Department of Pl...",cultural adaptation and validation of the life...
2,218.102.031-1.0IN,NO,,PROMOVIENDO LA REFLEXIÓN EN ESTUDIANTES DE PRE...,,EL PRESENTE PROYECTO INVOLUCRA LA REALIZACIÓN ...,FACULTAD DE ODONTOLOGÍA,DEPARTAMENTO DE ASTRONOMÍA,promoting reflection in preclinical dental stu...,the present project involves the realization o...,,faculty of dentistry,Department of Astronomy,promoting reflection in preclinical dental stu...
3,218.163.016-INI,INDEFINIDO,,MOTIVACIÓN Y HABILIDADES SOCIALES EN ADOLESCENTES,,EL ESTUDIO DE LA MOTIVACIÓN TIENE DIFERENTES A...,FACULTAD DE EDUCACIÓN,"DEPARTAMENTO DE CIENCIAS DE LA EDUCACIÓN, DEPT...",motivation and social skills in adolescents,the study of the motivation has different side...,,Faculty of Education,"Department of Education Sciences, Department o...",motivation and social skills in adolescents t...
4,219.091.052-INI,NO,,TIME EFFECTS ON THE LIQUEFACTION RESPONSE OF G...,,SECONDARY CONSOLIDATION AND AGEING ARE TWO OFT...,FACULTAD DE INGENIERÍA,DEPTO. TEORÍA POLITICA Y FUND.DE LA EDUC.,time effects on the liquefaction response of g...,secondary consolidation and ageing are two oft...,,Faculty of Engineering,Department of Political and Fund Theory of Edu...,time effects on the liquefaction response of g...
